In [ ]:
# Name: Nuraleya Binti Azizan
# ID: IS01084475

In [28]:
# For text preprocessing
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
# For topic modeling
from gensim import corpora
from gensim.models import LdaModel
import pandas as pd
from gensim.models.coherencemodel import CoherenceModel

In [29]:
#Load the data
df = pd.read_csv('news_dataset.csv')
df = df[df['text'].notnull()].copy() #remove null values
documents = df['text'].tolist()

In [30]:
#preprocess data
stop_words = set(stopwords.words('english')) # Create a set of English stopwords
lemmatizer = WordNetLemmatizer() # Initialize a WordNet lemmatizer
def preprocess_text(text):
 tokens = (word_tokenize(text.lower())) # Tokenize the text into words and convert to lowercase
 tokens = [token for token in tokens if token.isalnum()] # Filter out non-alphanumeric tokens
 tokens = [token for token in tokens if token not in stop_words] # Remove stopwords from the tokens
 tokens = [lemmatizer.lemmatize(token) for token in tokens] # Lemmatize each token
 return tokens # Return the preprocessed tokens
preprocessed_documents = [preprocess_text(doc) for doc in documents] # Preprocess each document in the list
print(preprocessed_documents[0])

['wondering', 'anyone', 'could', 'enlighten', 'car', 'saw', 'day', 'sport', 'car', 'looked', 'late', 'early', '70', 'called', 'bricklin', 'door', 'really', 'small', 'addition', 'front', 'bumper', 'separate', 'rest', 'body', 'know', 'anyone', 'tellme', 'model', 'name', 'engine', 'spec', 'year', 'production', 'car', 'made', 'history', 'whatever', 'info', 'funky', 'looking', 'car', 'please']


In [31]:
# Create a Gensim Dictionary object from the preprocessed documents
dictionary = corpora.Dictionary(preprocessed_documents)
# Filter out tokens that appear in less than 15 documents or more than 50% of the documents
dictionary.filter_extremes(no_below=15, no_above=0.5)
# Convert each preprocessed document into a bag-of-words representation using the dictionary
corpus = [dictionary.doc2bow(doc) for doc in preprocessed_documents] 

In [32]:
# Run LDA
lda_model = LdaModel(corpus, num_topics=4, id2word=dictionary, passes=15) # Train an LDA model on the corpus with 4 topics using Gensim's LdaModel class

In [34]:
# Evaluate LDA model using coherence score

coherence_model_lda = CoherenceModel(model=lda_model, texts=preprocessed_documents, dictionary=dictionary, coherence='c_v')
coherence_lda = coherence_model_lda.get_coherence()
print(f'Coherence Score: {coherence_lda:.4f}')
print()

Coherence Score: 0.6557



In [35]:
# empty list to store dominant topic labels for each document
article_labels = []
# iterate over each processed document
for i, doc in enumerate(preprocessed_documents):
 # for each document, convert to bag-of-words representation
 bow = dictionary.doc2bow(doc)
 # get list of topic probabilities
 topics = lda_model.get_document_topics(bow)
 # determine topic with highest probability
 dominant_topic = max(topics, key=lambda x: x[1])[0]
 # append to the list
 article_labels.append(dominant_topic)

# Create DataFrame
df_result = pd.DataFrame({"Article": documents, "Topic": article_labels})

# Print the DataFrame
print("Table with Articles and Topic:")
print(df_result)
print()

# Print Top Terms for each topic for interpretation
for topic_id in range(lda_model.num_topics): 
 print(f"Top terms for Topic #{topic_id}:") 
 top_terms = lda_model.show_topic(topic_id, topn=10) 
 print([term[0] for term in top_terms]) 
print()

Table with Articles and Topic:
                                                 Article  Topic
0      I was wondering if anyone out there could enli...      0
1      I recently posted an article asking what kind ...      0
2      \nIt depends on your priorities.  A lot of peo...      0
3      an excellent automatic can be found in the sub...      3
4      : Ford and his automobile.  I need information...      0
...                                                  ...    ...
11091  Secrecy in Clipper Chip\n\nThe serial number o...      3
11092  Hi !\n\nI am interested in the source of FEAL ...      3
11093  The actual algorithm is classified, however, t...      1
11094  \n\tThis appears to be generic calling upon th...      0
11095  \nProbably keep quiet and take it, lest they g...      0

[11096 rows x 2 columns]

Top terms for Topic #0:
['would', 'one', 'people', 'think', 'know', 'say', 'like', 'time', 'get', 'god']
Top terms for Topic #1:
['key', 'government', 'state', 'president', '

In [37]:
# Print the top terms for each topic with weight
print("Top Terms for Each Topic:")
for idx, topic in lda_model.print_topics():
 print(f"Topic {idx}:")
 terms = [term.strip() for term in topic.split("+")]
 for term in terms:
  weight, word = term.split("*")
  print(f"- {word.strip()} (weight: {weight.strip()})")
 print()


Top Terms for Each Topic:
Topic 0:
- "would" (weight: 0.012)
- "one" (weight: 0.011)
- "people" (weight: 0.009)
- "think" (weight: 0.008)
- "know" (weight: 0.007)
- "say" (weight: 0.006)
- "like" (weight: 0.006)
- "time" (weight: 0.005)
- "get" (weight: 0.005)
- "god" (weight: 0.005)

Topic 1:
- "key" (weight: 0.010)
- "government" (weight: 0.009)
- "state" (weight: 0.006)
- "president" (weight: 0.006)
- "encryption" (weight: 0.006)
- "people" (weight: 0.005)
- "new" (weight: 0.005)
- "law" (weight: 0.005)
- "q" (weight: 0.005)
- "system" (weight: 0.004)

Topic 2:
- "1" (weight: 0.056)
- "x" (weight: 0.043)
- "0" (weight: 0.042)
- "max" (weight: 0.040)
- "2" (weight: 0.039)
- "q" (weight: 0.030)
- "7" (weight: 0.027)
- "g" (weight: 0.026)
- "r" (weight: 0.026)
- "3" (weight: 0.025)

Topic 3:
- "file" (weight: 0.008)
- "use" (weight: 0.008)
- "one" (weight: 0.007)
- "x" (weight: 0.007)
- "db" (weight: 0.007)
- "system" (weight: 0.007)
- "get" (weight: 0.006)
- "window" (weight: 0.006)
-

In [ ]:
''' Interpretation on Coherence Score: 

The model achieved a coherence score of 0.6557. 
In topic modeling, a score above 0.5 is generally considered a strong indicator 
that the topics are distinct and accurate. This score validates 
that the 4 topics identified are semantically consistent.

_______________________________________________________________________________________________
Topic Analysis:

Topic 0 (Philosophical/Religion): Dominated by words like people, think, god, and know. 
This topic represents discussions regarding philosophy or religion.

Topic 1 (Government & Security): defined by government, state, president, and encryption. 
This topic is news related to government's security.

Topic 2 (Noise): Consists of numbers and single characters (1, x, 0, max). 
This might be due to raw datasets that contain encoded data which can be seen as noise.

Topic 3 (IT): Focused on terms like file, use, db, and window. 
This topic clusters news related to software and database management.

'''